<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/02_construction_data_preparation_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# การเตรียมข้อมูลรายการจ้างก่อสร้าง ปีงบประมาณ 2569

ข้อมูล e-GP ปีงบประมาณ 2569 สะสมถึงวันที่ 30 กรกฎาคม 2569 จากไฟล์ต้นทาง 8 ไฟล์

Notebook นี้รวมไฟล์ สกัดรายการ `จ้างก่อสร้าง` และตรวจข้อมูลเบื้องต้นก่อนนำไปสำรวจต่อ


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [ ]:
base_dir = Path('/content/drive/MyDrive/learning/dads/dads5001/project_1_dads5001/dataset/procurement/egp-contract/2569')

processed_dir = base_dir.parent / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(base_dir.glob('*.csv'))
construction_path = processed_dir / 'construction_contracts_2569.csv'

project_dir = base_dir.parents[3]
figure_dir = project_dir / 'figure'
figure_dir.mkdir(parents=True, exist_ok=True)

print(f'CSV files: {len(csv_files)}')
print(f'Output file: {construction_path}')

## 1. ตรวจโครงสร้างไฟล์ต้นทาง

อ่านตัวอย่างจากไฟล์แรกเพื่อตรวจชื่อคอลัมน์และรูปแบบข้อมูล


In [ ]:
sample_data = pd.read_csv(csv_files[0], nrows=5)

print(f'Sample file: {csv_files[0].name}')
print(f'Columns: {sample_data.shape[1]}')

display(sample_data)

In [ ]:
for number, column in enumerate(sample_data.columns, start=1):
    print(f'{number:02d}. {column}')

## 2. รวมไฟล์และสกัดรายการจ้างก่อสร้าง

อ่านข้อมูลทีละไฟล์ นับประเภทโครงการ และเก็บเฉพาะรายการจ้างก่อสร้าง


In [ ]:
project_type_column = 'ชื่อประเภทโครงการ'

type_counts_list = []
construction_parts = []
file_summaries = []

for file_path in csv_files:
    data = pd.read_csv(file_path, low_memory=False)
    data.columns = data.columns.str.strip()

    project_type = data[project_type_column].astype('string')
    project_type = project_type.str.strip().fillna('ไม่ระบุ')

    type_counts_list.append(project_type.value_counts())

    construction = data[project_type == 'จ้างก่อสร้าง'].copy()
    construction['source_file'] = file_path.name
    construction_parts.append(construction)

    file_summaries.append({
        'file_name': file_path.name,
        'total_rows': len(data),
        'construction_rows': len(construction)
    })

    print(f'{file_path.name}: {len(data):,} rows | 'f'{len(construction):,} construction rows')

In [ ]:
processing_summary = pd.DataFrame(file_summaries)
total_records = processing_summary['total_rows'].sum()

construction_data = pd.concat(
    construction_parts,
    ignore_index=True
)

total_construction_records = len(construction_data)
construction_pct = total_construction_records / total_records * 100

print(f'Total records: {total_records:,}')
print(f'Construction records: {total_construction_records:,}')
print(f'Construction share: {construction_pct:.2f}%')


### สัดส่วนรายการตามประเภทโครงการ

เปรียบเทียบจำนวนรายการของแต่ละประเภทกับข้อมูลทั้งหมด


In [ ]:
type_counts = pd.concat(type_counts_list, axis=1).fillna(0)
type_counts = type_counts.sum(axis=1).astype('int64')
type_counts = type_counts.sort_values(ascending=False)

project_type_counts = type_counts.reset_index()
project_type_counts.columns = [project_type_column, 'record_count']

project_type_counts['record_pct'] = (project_type_counts['record_count'] / total_records * 100)
display(project_type_counts)

In [ ]:
# ดาวน์โหลดฟอนต์สำหรับแสดงภาษาไทยในกราฟ
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf

fm.fontManager.addfont('thsarabunnew-webfont.ttf')
sns.set_theme(style='whitegrid', font='TH Sarabun New')
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
plot_data = project_type_counts.head(4).copy()
plot_data = plot_data.sort_values('record_pct')

colors = []
for project_type in plot_data[project_type_column]:
    if project_type == 'จ้างก่อสร้าง':
        colors.append('#4C6A85')
    else:
        colors.append('#A7B1BB')

In [ ]:
ax = plot_data.plot.barh(
    x=project_type_column,
    y='record_pct',
    color=colors,
    figsize=(11, 5.5),
    legend=False
)

ax.set_title(
    f'งานจ้างก่อสร้างคิดเป็น {construction_pct:.2f}% ของรายการจัดซื้อจัดจ้าง',
    loc='left',
    fontsize=18,
    fontweight='bold'
)
ax.set_xlabel('สัดส่วนของจำนวนรายการ (%)')
ax.set_ylabel('')
ax.grid(axis='x', color='#E5E9ED')
ax.grid(axis='y', visible=False)

for bar, percentage in zip(ax.patches, plot_data['record_pct']):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f'{percentage:.2f}%',
        va='center'
    )

sns.despine(left=True, bottom=True)
fig = ax.get_figure()
fig.tight_layout()


In [ ]:
png_path = figure_dir / 'fig02_01_procurement_to_construction.png'
svg_path = figure_dir / 'fig02_01_procurement_to_construction.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight')
fig.savefig(svg_path, bbox_inches='tight')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')

### ผลการสกัด

พบรายการจ้างก่อสร้าง 180,079 รายการ จากทั้งหมด 3,964,924 รายการ หรือ 4.54% จำนวนนี้เป็นจำนวนแถวในไฟล์ต้นทาง ยังไม่ใช่จำนวนโครงการไม่ซ้ำ


## 3. ตรวจโครงสร้างข้อมูลจ้างก่อสร้าง

ตรวจจำนวนโครงการ เลขที่สัญญา แถวซ้ำ และโครงการที่ปรากฏมากกว่าหนึ่งแถว


In [ ]:
project_id_column = 'รหัสโครงการ'
contract_column = 'เลขที่สัญญา'

duplicate_rows = construction_data.drop(
    columns='source_file'
).duplicated().sum()

project_contract_count = construction_data[
    [project_id_column, contract_column]
].drop_duplicates().shape[0]

data_summary = {
    'รายการจ้างก่อสร้าง': len(construction_data),
    'โครงการไม่ซ้ำ': construction_data[project_id_column].nunique(),
    'คู่โครงการ–สัญญาไม่ซ้ำ': project_contract_count,
    'รหัสโครงการที่หาย': construction_data[project_id_column].isna().sum(),
    'เลขที่สัญญาที่หาย': construction_data[contract_column].isna().sum(),
    'แถวซ้ำทุกคอลัมน์': duplicate_rows
}

display(pd.Series(data_summary, name='value').to_frame())


In [ ]:
project_row_counts = construction_data[project_id_column].value_counts()

repeated_project_count = (project_row_counts > 1).sum()
maximum_rows = project_row_counts.max()

print(f'โครงการที่มีมากกว่า 1 แถว: {repeated_project_count:,}')
print(f'จำนวนแถวสูงสุดต่อโครงการ: {maximum_rows:,}')


In [ ]:
repeated_project_ids = project_row_counts[
    project_row_counts > 1
].head(3).index

columns_to_show = [
    project_id_column,
    'ชื่อโครงการจัดซื้อจัดจ้าง',
    'เลขประจำตัวนิติบุคคล 13 หลัก',
    'ชื่อผู้ชนะการเสนอราคา',
    contract_column,
    'วงเงินงบประมาณในสัญญา (บาท)'
]

repeated_project_sample = construction_data[
    construction_data[project_id_column].isin(repeated_project_ids)
]

display(repeated_project_sample[columns_to_show])


### จัดข้อมูลให้อยู่ในระดับสัญญา

โครงการหนึ่งอาจมีหลายเลขที่สัญญา ซึ่งต้องเก็บไว้ทั้งหมด แต่ถ้าคู่ `รหัสโครงการ–เลขที่สัญญา` ซ้ำ จะเก็บเพียงแถวแรก

แถวซ้ำระดับสัญญาบางส่วนเกิดจากการบันทึกกิจการค้าร่วมและสมาชิกแยกคนละแถว จึงไม่สรุปว่าทุกกรณีเป็นกิจการค้าร่วม


In [ ]:
contract_key = [
    project_id_column,
    contract_column
]

duplicate_contract_mask = construction_data.duplicated(
    subset=contract_key,
    keep=False
)

duplicate_contract_rows = construction_data[
    duplicate_contract_mask
].sort_values(contract_key)

duplicate_contract_groups = duplicate_contract_rows[
    contract_key
].drop_duplicates()

print(f'แถวที่อยู่ในคู่โครงการ–สัญญาซ้ำ: {len(duplicate_contract_rows):,}')
print(f'คู่โครงการ–สัญญาที่ซ้ำ: {len(duplicate_contract_groups):,}')

display(duplicate_contract_rows[columns_to_show].head(30))


In [ ]:
construction_contract_data = construction_data.drop_duplicates(
    subset=contract_key,
    keep='first'
).copy()

removed_rows = len(construction_data) - len(construction_contract_data)

contract_count_by_project = construction_contract_data.groupby(
    project_id_column
)[contract_column].nunique()

multiple_contract_projects = (
    contract_count_by_project > 1
).sum()

print(f'รายการก่อนจัดข้อมูล: {len(construction_data):,}')
print(f'รายการระดับสัญญา: {len(construction_contract_data):,}')
print(f'แถวที่ตัดออก: {removed_rows:,}')
print(f'โครงการที่มีหลายเลขที่สัญญา: {multiple_contract_projects:,}')


In [ ]:
construction_contract_data.to_csv(
    construction_path,
    index=False,
    encoding='utf-8-sig'
)

print(f'Saved to: {construction_path}')


### ความพร้อมของฟิลด์ที่ใช้วิเคราะห์

ตรวจค่าที่หายของจังหวัด หน่วยงานย่อย ผู้รับจ้าง และวันที่เกิดรายการในระดับโครงการ


In [ ]:
province_column = 'จังหวัด'
subagency_column = 'ชื่อหน่วยงานย่อย'
supplier_id_column = 'เลขประจำตัวนิติบุคคล 13 หลัก'
supplier_name_column = 'ชื่อผู้ชนะการเสนอราคา'
transaction_date_column = 'วันที่เกิดรายการ'

quality_columns = [
    province_column,
    subagency_column,
    supplier_id_column,
    supplier_name_column,
    transaction_date_column
]

for column in quality_columns:
    construction_contract_data[column] = construction_contract_data[column].astype('string')
    construction_contract_data[column] = construction_contract_data[column].str.strip()
    construction_contract_data[column] = construction_contract_data[column].replace('', pd.NA)

In [ ]:
project_data = construction_contract_data.drop_duplicates(
    subset=project_id_column,
    keep='first'
).copy()


In [ ]:
province_check = construction_contract_data[
    province_column
].value_counts(dropna=False)

display(province_check.to_frame(name='record_count'))


## ไฟล์ผลลัพธ์

ไฟล์ `construction_contracts_2569.csv` เก็บหนึ่งแถวต่อคู่ `รหัสโครงการ–เลขที่สัญญา` โดยยังคงทุกเลขที่สัญญาที่ต่างกันภายในโครงการเดียวกัน พร้อมนำไปสำรวจต่อใน Notebook 03
